# Analyse Exploratoire — Projet Data Centre d'Assistance
## DU Big Data & Data Science — Université de Montpellier 2025-2026
### Équipe : Randriamisaina Tsiory-Fanomezana · SHIRALI POUR Amir

Ce notebook réalise l'analyse exploratoire complète du dataset final (`dataset_complet.csv`).

**Plan :**
1. Chargement et aperçu global
2. Analyse univariée — variables numériques (`analyse_var`)
3. Analyse univariée — variables catégorielles
4. Analyse bivariée — durée vs variables explicatives
5. Analyse temporelle et saisonnalité
6. Synthèse et conclusions

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Détection de la racine du projet
def _racine():
    rep = Path().resolve()
    while True:
        if any((rep / m).exists() for m in ['.git', 'requirements.txt']):
            return rep
        if rep.parent == rep:
            return Path().resolve().parent
        rep = rep.parent

RACINE = _racine()
print(f'Racine du projet : {RACINE}')

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
sns.set_style('whitegrid')
BLEU = '#4472C4'; ORANGE = '#ED7D31'; VERT = '#70AD47'; ROUGE = '#C00000'

## 1. Chargement et aperçu global

In [ ]:
df = pd.read_csv(RACINE / 'data' / 'dataset_complet.csv', encoding='utf-8')
df['date.ouverture'] = pd.to_datetime(df['date.ouverture'], errors='coerce')

print(f'Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Période : {df["date.ouverture"].dt.year.min()} – {df["date.ouverture"].dt.year.max()}')
print(f'Agents uniques : {df["Matricule.de.traitement"].nunique():,}')
df.head(3)

In [ ]:
# Taux de valeurs manquantes par colonne
nan_pct = (df.isna().mean() * 100).sort_values(ascending=False)
nan_pct = nan_pct[nan_pct > 0]
print('Colonnes avec valeurs manquantes :')
print(nan_pct.to_string())

## 2. Analyse univariée — variables numériques

Fonction `analyse_var` : stats descriptives + histogramme + boxplot + test de normalité.

In [ ]:
def analyse_var(df, var, titre=None):
    """Analyse univariée complète d'une variable numérique."""
    serie = df[var].dropna()
    titre = titre or var

    print(f'=== {titre} ({len(serie):,} valeurs non-NaN) ===')
    print(serie.describe().round(2).to_string())
    print(f'  Asymétrie (skewness) : {serie.skew():.3f}')
    print(f'  Kurtosis             : {serie.kurt():.3f}')

    # Test de normalité (Shapiro sur échantillon)
    sample = serie.sample(min(5000, len(serie)), random_state=42)
    stat, p = stats.shapiro(sample)
    print(f'  Shapiro-Wilk         : W={stat:.4f}, p={p:.4e} → {"NON normale" if p < 0.05 else "normale"}')

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(titre, fontsize=13, fontweight='bold')

    # Histogramme
    p99 = serie.quantile(0.99)
    axes[0].hist(serie[serie <= p99], bins=50, color=BLEU, edgecolor='white')
    axes[0].axvline(serie.median(), color=ROUGE, linestyle='--', label=f'Médiane={serie.median():.1f}')
    axes[0].axvline(serie.mean(), color=ORANGE, linestyle='--', label=f'Moyenne={serie.mean():.1f}')
    axes[0].set_title('Distribution (p99)')
    axes[0].legend(fontsize=9)

    # Boxplot
    axes[1].boxplot(serie, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=BLEU, alpha=0.7))
    axes[1].set_title('Boxplot')
    axes[1].set_ylabel(var)

    # QQ-plot
    stats.probplot(sample, dist='norm', plot=axes[2])
    axes[2].set_title('QQ-plot (normalité)')

    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# Variables numériques à analyser
vars_num = [
    ('duree_totale_h',              'Durée totale de traitement (heures)'),
    ('agent_experience_j',          'Expérience de l\'agent (jours)'),
    ('agent_duree_travail_j',        'Durée de travail de l\'agent (jours)'),
    ('agent_temps_travail_pct',      'Taux de temps de travail (%)'),
    ('nb_interventions',             'Nombre d\'interventions par dossier'),
    ('nb_agents_distincts',          'Nombre d\'agents distincts par dossier'),
    ('delai_survenance_ouverture_j', 'Délai survenance → ouverture (jours)'),
]

for var, titre in vars_num:
    if var in df.columns:
        analyse_var(df, var, titre)

## 3. Analyse univariée — variables catégorielles

In [ ]:
def analyse_cat(df, var, titre=None, top_n=15):
    """Analyse univariée d'une variable catégorielle."""
    serie = df[var].dropna()
    titre = titre or var
    vc = serie.value_counts()

    print(f'=== {titre} ===')
    print(f'  Modalités : {serie.nunique()}')
    print(f'  Top 5 :\n{vc.head().to_string()}')

    fig, ax = plt.subplots(figsize=(10, 4))
    vc.head(top_n).sort_values().plot(kind='barh', ax=ax, color=BLEU, edgecolor='white')
    ax.set_title(f'Distribution — {titre}', fontweight='bold')
    ax.set_xlabel('Nombre de dossiers')
    for i, v in enumerate(vc.head(top_n).sort_values()):
        ax.text(v + len(df)*0.002, i, f'{v/len(df)*100:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    print()

vars_cat = [
    ('Cause.intervention',   'Cause d\'intervention'),
    ('Formule',              'Formule du contrat'),
    ('agent_contrat',        'Type de contrat agent'),
    ('agent_lieu_travail',   'Lieu de travail agent'),
    ('agent_population',     'Population agent'),
    ('Client',               'Client'),
    ('Type.d.energie',       'Type d\'énergie'),
]

for var, titre in vars_cat:
    if var in df.columns:
        analyse_cat(df, var, titre)

## 4. Analyse bivariée — Durée de traitement vs variables explicatives

In [ ]:
duree_valide = df.dropna(subset=['duree_totale_h'])

# 4.1 Durée par Cause d'intervention — boxplot + test Kruskal-Wallis
fig, ax = plt.subplots(figsize=(12, 5))
causes_order = (duree_valide.groupby('Cause.intervention')['duree_totale_h']
                .median().sort_values(ascending=False).index)
duree_valide.boxplot(column='duree_totale_h', by='Cause.intervention',
                     ax=ax, order=causes_order, showfliers=False,
                     boxprops=dict(color=BLEU), medianprops=dict(color=ROUGE))
ax.set_title('Durée de traitement par Cause d\'intervention', fontweight='bold')
ax.set_xlabel('Cause d\'intervention')
ax.set_ylabel('Durée (heures)')
ax.tick_params(axis='x', rotation=45)
plt.suptitle('')
plt.tight_layout()
plt.show()

# Test statistique Kruskal-Wallis
groupes = [g['duree_totale_h'].values for _, g in duree_valide.groupby('Cause.intervention')]
h_stat, p_val = stats.kruskal(*groupes)
print(f'Test Kruskal-Wallis : H={h_stat:.2f}, p={p_val:.4e}')
print('→ Différences significatives entre causes' if p_val < 0.05 else '→ Pas de différence significative')

In [ ]:
# 4.2 Durée par type de contrat agent
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contrats = duree_valide.groupby('agent_contrat')['duree_totale_h']
medians = contrats.median().sort_values(ascending=False)
axes[0].bar(medians.index, medians.values, color=BLEU, edgecolor='white')
axes[0].set_title('Durée médiane par type de contrat', fontweight='bold')
axes[0].set_ylabel('Durée médiane (h)')
axes[0].tick_params(axis='x', rotation=30)

# Test Mann-Whitney CDI vs CDD
if 'CDI' in duree_valide['agent_contrat'].values and 'CDD' in duree_valide['agent_contrat'].values:
    cdi = duree_valide[duree_valide['agent_contrat'] == 'CDI']['duree_totale_h']
    cdd = duree_valide[duree_valide['agent_contrat'] == 'CDD']['duree_totale_h']
    u, p = stats.mannwhitneyu(cdi, cdd, alternative='two-sided')
    axes[0].set_xlabel(f'Mann-Whitney CDI vs CDD : p={p:.4e}')

# Lieu de travail SITE vs TELE
lieu = duree_valide.groupby('agent_lieu_travail')['duree_totale_h'].median()
axes[1].bar(lieu.index, lieu.values, color=[BLEU, ORANGE], edgecolor='white')
axes[1].set_title('Durée médiane : Site vs Télétravail', fontweight='bold')
axes[1].set_ylabel('Durée médiane (h)')
for i, v in enumerate(lieu.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}h', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Expérience agent vs durée — scatter + corrélation de Spearman
sample = duree_valide.dropna(subset=['agent_experience_j']).sample(min(5000, len(duree_valide)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(sample['agent_experience_j'], sample['duree_totale_h'],
                alpha=0.2, s=6, color=BLEU)
axes[0].set_xlabel('Expérience agent (jours)')
axes[0].set_ylabel('Durée totale (h)')
axes[0].set_title('Expérience vs Durée de traitement', fontweight='bold')

rho, p = stats.spearmanr(sample['agent_experience_j'], sample['duree_totale_h'])
axes[0].text(0.05, 0.95, f'ρ Spearman={rho:.3f}\np={p:.3e}',
             transform=axes[0].transAxes, va='top', bbox=dict(boxstyle='round', facecolor='wheat'))

# Nb interventions vs durée
axes[1].scatter(sample['nb_interventions'], sample['duree_totale_h'],
                alpha=0.2, s=6, color=ORANGE)
axes[1].set_xlabel('Nombre d\'interventions')
axes[1].set_ylabel('Durée totale (h)')
axes[1].set_title('Nb interventions vs Durée', fontweight='bold')

rho2, p2 = stats.spearmanr(sample['nb_interventions'], sample['duree_totale_h'])
axes[1].text(0.05, 0.95, f'ρ Spearman={rho2:.3f}\np={p2:.3e}',
             transform=axes[1].transAxes, va='top', bbox=dict(boxstyle='round', facecolor='wheat'))

plt.tight_layout()
plt.show()